# Yelp Restaurants EDA

This notebook loads parsed parquet files from `data/processed/` and explores:

1. Category distribution and long-tail behavior
2. Rating and review-count distributions
3. `price_range` distribution and missingness
4. Geographic spread by city
5. Correlations across numerical features

All plots use Plotly and are intended to run on a fresh kernel.

In [ ]:
from pathlib import Path
import math
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown

# Resolve project root from notebook location.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BUSINESS_PATH = PROCESSED_DIR / "business_restaurants.parquet"
REVIEW_PATH = PROCESSED_DIR / "review_restaurants.parquet"
USER_PATH = PROCESSED_DIR / "user_restaurants.parquet"

for p in [BUSINESS_PATH, REVIEW_PATH, USER_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required parquet file: {p}")

business_df = pd.read_parquet(BUSINESS_PATH)
review_df = pd.read_parquet(REVIEW_PATH)
user_df = pd.read_parquet(USER_PATH)

print("Loaded shapes:")
print(f"business: {business_df.shape}")
print(f"review:   {review_df.shape}")
print(f"user:     {user_df.shape}")

## 1) Category Distribution and Long Tail

We split business categories, keep top cuisines, and quantify category concentration.

Long-tail metric used here:
- Sort categories by restaurant coverage
- Find how many categories are needed to cover 80% of restaurants (a restaurant is covered if it has at least one selected category)

In [ ]:
def _split_categories(value):
    if pd.isna(value):
        return []
    parts = [c.strip() for c in str(value).split(",") if c and c.strip()]
    return parts

if "business_id" not in business_df.columns:
    raise ValueError("business_restaurants.parquet must contain business_id")

cat_frame = business_df[["business_id", "categories"]].copy() if "categories" in business_df.columns else business_df[["business_id"]].assign(categories="")
cat_frame["category_list"] = cat_frame["categories"].apply(_split_categories)
exploded = cat_frame.explode("category_list").rename(columns={"category_list": "category"})
exploded = exploded.dropna(subset=["category"])
exploded["category"] = exploded["category"].astype(str).str.strip()
exploded = exploded[exploded["category"] != ""]

# Optional cleanup: remove broad bucket so cuisine-specific patterns are easier to see.
exploded = exploded[exploded["category"].str.lower() != "restaurants"]

category_counts = (
    exploded.groupby("category")["business_id"]
    .nunique()
    .sort_values(ascending=False)
)

top30 = category_counts.head(30).reset_index(name="restaurant_count")
fig_top30 = px.bar(
    top30,
    x="restaurant_count",
    y="category",
    orientation="h",
    title="Top 30 Categories by Restaurant Count",
)
fig_top30.update_layout(yaxis={"categoryorder": "total ascending"})
fig_top30.show()

# Long-tail coverage over unique restaurants.
cat_to_business = exploded.groupby("category")["business_id"].agg(lambda s: set(s))
all_restaurants = set(cat_frame["business_id"].dropna().astype(str).tolist())
covered = set()

categories_for_80 = 0
coverage_at_80 = 0.0
for cat in category_counts.index:
    covered |= {str(v) for v in cat_to_business[cat]}
    categories_for_80 += 1
    coverage = (len(covered) / len(all_restaurants)) if all_restaurants else 0.0
    if coverage >= 0.80:
        coverage_at_80 = coverage
        break

if categories_for_80 == 0 and len(category_counts) > 0:
    categories_for_80 = len(category_counts)
    coverage_at_80 = (len(covered) / len(all_restaurants)) if all_restaurants else 0.0

display(Markdown(
    f"**Long-tail result:** `{categories_for_80}` categories cover "
    f"`{coverage_at_80*100:.2f}%` of restaurants (target: 80%)."
))

display(Markdown(
    "**CTR feature signal:** category/cuisine is a strong intent feature. "
    "Use multi-hot or learned embeddings; group rare categories into an `OTHER` bucket."
))

## 2) Rating and Review Count Distributions

We inspect `stars` and `review_count` from business-level data.
- Ratings are often left-skewed in Yelp-like data.
- Review counts are typically heavy-tailed (power-law-like), so `log1p` is useful.

In [ ]:
if "stars" in business_df.columns:
    fig_stars = px.histogram(
        business_df,
        x="stars",
        nbins=25,
        title="Restaurant Rating Distribution (stars)",
    )
    fig_stars.show()
else:
    display(Markdown("`stars` not found in business parquet."))

if "review_count" in business_df.columns:
    fig_reviews = px.histogram(
        business_df,
        x="review_count",
        nbins=80,
        title="Review Count Distribution (Raw)",
    )
    fig_reviews.show()

    review_count_log = np.log1p(business_df["review_count"].fillna(0))
    fig_reviews_log = px.histogram(
        x=review_count_log,
        nbins=80,
        title="Review Count Distribution (log1p transformed)",
        labels={"x": "log1p(review_count)"},
    )
    fig_reviews_log.show()

    display(Markdown(
        "**CTR feature signal:** `stars` and `review_count` are useful trust/popularity signals. "
        "Apply scaling and log transform to stabilize heavy tails."
    ))
else:
    display(Markdown("`review_count` not found in business parquet."))

## 3) Price Range Distribution and Missingness

We extract `price_range` from Yelp attributes (`RestaurantsPriceRange2`) and quantify missing values.

In [ ]:
def extract_price_range(attributes_value):
    if attributes_value is None or (isinstance(attributes_value, float) and math.isnan(attributes_value)):
        return np.nan

    # Usually a dict loaded from JSON lines.
    if isinstance(attributes_value, dict):
        raw = attributes_value.get("RestaurantsPriceRange2")
    else:
        raw = None

    if raw is None:
        return np.nan

    raw_str = str(raw).strip().strip("'").strip('"')
    if raw_str == "":
        return np.nan

    try:
        return int(float(raw_str))
    except Exception:
        return np.nan

if "attributes" in business_df.columns:
    business_df["price_range"] = business_df["attributes"].apply(extract_price_range)
else:
    business_df["price_range"] = np.nan

price_missing_pct = business_df["price_range"].isna().mean() * 100
price_counts = (
    business_df["price_range"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("price_range")
    .reset_index(name="restaurant_count")
)

if not price_counts.empty:
    fig_price = px.bar(
        price_counts,
        x="price_range",
        y="restaurant_count",
        title="Price Range Distribution (RestaurantsPriceRange2)",
    )
    fig_price.show()
else:
    display(Markdown("No non-missing `price_range` values found."))

display(Markdown(
    f"**Missing `price_range`:** `{price_missing_pct:.2f}%` of restaurants are missing this feature."
))

display(Markdown(
    "**CTR preprocessing note:** keep a dedicated missing indicator for `price_range` and impute missing values "
    "(e.g., median or model-based), since missingness itself may carry signal."
))

## 4) Geographic Spread

We examine top cities by restaurant count to understand market concentration and regional coverage.

In [ ]:
if "city" in business_df.columns:
    city_counts = (
        business_df["city"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
        .value_counts()
        .head(20)
        .rename_axis("city")
        .reset_index(name="restaurant_count")
    )

    fig_city = px.bar(
        city_counts,
        x="restaurant_count",
        y="city",
        orientation="h",
        title="Top 20 Cities by Restaurant Count",
    )
    fig_city.update_layout(yaxis={"categoryorder": "total ascending"})
    fig_city.show()

    display(Markdown(
        "**CTR feature signal:** city/location is useful for regional behavior differences and local competition effects."
    ))
else:
    display(Markdown("`city` not found in business parquet."))

## 5) Correlation Matrix (Numerical Features)

We compute correlations among numeric business features to identify redundancy and potential interactions.

In [ ]:
candidate_numeric = [
    "stars",
    "review_count",
    "price_range",
    "latitude",
    "longitude",
    "is_open",
]
existing_numeric = [c for c in candidate_numeric if c in business_df.columns]

if existing_numeric:
    corr_df = business_df[existing_numeric].apply(pd.to_numeric, errors="coerce")
    corr = corr_df.corr(numeric_only=True)

    fig_corr = px.imshow(
        corr,
        text_auto=True,
        aspect="auto",
        color_continuous_scale="RdBu",
        zmin=-1,
        zmax=1,
        title="Correlation Matrix of Numerical Features",
    )
    fig_corr.show()
else:
    display(Markdown("No expected numerical features found for correlation analysis."))

## Key Findings and CTR Feature Readiness

### Useful for CTR modeling
- **Category/cuisine**: strong intent/context signal; encode as multi-hot or embeddings.
- **Stars**: quality proxy likely correlated with clicks.
- **Review count**: popularity/social proof signal (use `log1p`).
- **City/location**: captures regional demand and user preferences.
- **Price range**: strong user-segment and intent feature.

### Needs preprocessing
- **Category long tail**: collapse rare categories to `OTHER` or use frequency thresholding.
- **Review count**: heavy tail; use log transform and scaling.
- **Price range missingness**: include missing indicator + imputation.
- **Numerical scaling**: standardize selected numeric features for stable model training.

### Acceptance checks covered
- Missing `price_range` percentage is computed and documented in markdown output.
- Category long-tail is quantified by the number of categories needed to cover 80% of restaurants.
- Notebook cells are ordered to run end-to-end on a fresh kernel.